# Weather Data Retrieval and Processing

This notebook demonstrates how the application resolves a place name into geographic coordinates and retrieves hourly forecast data from the Open-Meteo APIs.

It explores the returned weather variables, converts the response into a pandas DataFrame, and prepares readable weather descriptions for later processing.


## 1. Import libraries

`requests` is used for HTTP communication with Open-Meteo and `pandas` is used to organize the returned forecast data.

In [1]:
import requests
import pandas as pd
from datetime import datetime, timedelta

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Geocode a location

The chatbot will receive natural-language locations such as **Munich** or **Berlin**. Before requesting weather data, the Weather Tool needs geographic coordinates.

This function uses the Open-Meteo Geocoding API to convert a location name into latitude, longitude, country and timezone.

In [2]:
def geocode_location(location):
    """
    Convert a location name into geographic coordinates
    using the Open-Meteo Geocoding API.
    """

    url = "https://geocoding-api.open-meteo.com/v1/search"

    params = {
        "name": location,
        "count": 1,
        "language": "en",
        "format": "json"
    }

    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()

    data = response.json()

    if "results" not in data or not data["results"]:
        raise ValueError(f"Location not found: {location}")

    result = data["results"][0]

    return {
        "name": result["name"],
        "latitude": result["latitude"],
        "longitude": result["longitude"],
        "country": result.get("country"),
        "timezone": result.get("timezone")
    }

## 3. Test location search

We use **Munich** as the first prototype location because it is one of the representative scenarios defined in initial design.

In [3]:
location = geocode_location("Munich")

location

{'name': 'Munich',
 'latitude': 48.13743,
 'longitude': 11.57549,
 'country': 'Germany',
 'timezone': 'Europe/Berlin'}

## 4. Build the Weather Tool

The Weather Tool requests hourly forecast variables needed by the project:

- Temperature
- Precipitation probability
- Precipitation
- Weather code
- Cloud cover
- Wind speed

The API response remains separate from the natural-language generation layer.

In [4]:
def get_weather_forecast(
    latitude,
    longitude,
    timezone="auto",
    forecast_days=3
):
    """
    Retrieve hourly weather forecast from Open-Meteo.
    """

    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": ",".join([
            "temperature_2m",
            "precipitation_probability",
            "precipitation",
            "weather_code",
            "cloud_cover",
            "wind_speed_10m"
        ]),
        "timezone": timezone,
        "forecast_days": forecast_days,
        "temperature_unit": "celsius",
        "wind_speed_unit": "kmh",
        "precipitation_unit": "mm"
    }

    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()

    return response.json()

## 5. Retrieve the Munich forecast

In [5]:
weather_data = get_weather_forecast(
    latitude=location["latitude"],
    longitude=location["longitude"],
    timezone=location["timezone"],
    forecast_days=3
)

print("API response sections:")
print(weather_data.keys())

API response sections:
dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly'])


## 6. Convert the hourly response to a DataFrame

A DataFrame makes the forecast easier to inspect and later process for activity-specific weather logic.

In [6]:
hourly = weather_data["hourly"]

forecast_df = pd.DataFrame(hourly)

forecast_df.head(10)

,time,temperature_2m,precipitation_probability,precipitation,weather_code,cloud_cover,wind_speed_10m
0,2026-09-24T00:00,10.1,0,0.0,0,0,4.7
1,2026-09-24T01:00,10.8,0,0.0,0,0,6.0
2,2026-09-24T02:00,11.6,0,0.0,3,99,5.8
3,2026-09-24T03:00,12.3,0,0.0,3,99,8.8
4,2026-09-24T04:00,12.7,0,0.0,3,100,9.4
5,2026-09-24T05:00,13.0,0,0.0,3,100,8.4
6,2026-09-24T06:00,12.9,0,0.0,3,100,8.9
7,2026-09-24T07:00,13.3,0,0.0,3,100,10.6
8,2026-09-24T08:00,14.3,0,0.0,3,100,13.2
9,2026-09-24T09:00,15.1,3,0.0,3,100,13.7


## 7. Convert weather codes into readable descriptions

Open-Meteo returns a numerical weather code. The chatbot should present a readable description instead of exposing the numerical code directly.

In [7]:
WEATHER_CODES = {
    0: "Clear sky",
    1: "Mainly clear",
    2: "Partly cloudy",
    3: "Overcast",
    45: "Fog",
    48: "Depositing rime fog",
    51: "Light drizzle",
    53: "Moderate drizzle",
    55: "Dense drizzle",
    61: "Slight rain",
    63: "Moderate rain",
    65: "Heavy rain",
    71: "Slight snow",
    73: "Moderate snow",
    75: "Heavy snow",
    80: "Slight rain showers",
    81: "Moderate rain showers",
    82: "Violent rain showers",
    85: "Slight snow showers",
    86: "Heavy snow showers",
    95: "Thunderstorm",
    96: "Thunderstorm with slight hail",
    99: "Thunderstorm with heavy hail"
}


def weather_code_to_description(code):
    return WEATHER_CODES.get(
        int(code),
        "Unknown weather condition"
    )

## 8. Format one forecast record

This converts one API record into a concise human-readable weather summary.

In [8]:
def format_forecast_row(row):
    weather_description = weather_code_to_description(
        row["weather_code"]
    )

    return (
        f"Time: {row['time']}\n"
        f"Temperature: {row['temperature_2m']:.1f} °C\n"
        f"Conditions: {weather_description}\n"
        f"Wind: {row['wind_speed_10m']:.1f} km/h\n"
        f"Precipitation probability: "
        f"{row['precipitation_probability']:.0f}%\n"
        f"Precipitation: {row['precipitation']:.1f} mm"
    )

## 9. Display the first forecast record

In [9]:
print(format_forecast_row(forecast_df.iloc[0]))

Time: 2026-09-24T00:00
Temperature: 10.1 °C
Conditions: Clear sky
Wind: 4.7 km/h
Precipitation probability: 0%
Precipitation: 0.0 mm


## 10. Prototype validation

At this point, the Weather Tool should be able to complete the following pipeline:

**Location name → coordinates → Open-Meteo forecast → structured weather data**

The result contains the core information required by the application: temperature, sunny/cloudy conditions and wind speed. Precipitation is also retrieved for the extended chatbot use cases.

### Next implementation step

After this notebook works, the next notebook will implement **Activity Weather Logic**. It will interpret the same forecast differently for hiking, cycling, running, camping and mountain biking before we connect the LLM agent.